# AIoT-DA L6: 50 Startups 多元線性回歸分析與 5 大機器學習演算法評測
### 採用 CRISP-DM 跨產業資料探勘標準流程
---
本專案嚴格遵循 **CRISP-DM (Cross-Industry Standard Process for Data Mining)** 六大標準階段：
1. **Business Understanding (商業理解)**
2. **Data Understanding (資料理解)**
3. **Data Preparation (資料準備：One-Hot Encoding、避免 Dummy Variable Trap)**
4. **Modeling (模型建立：多元線性回歸 + 5 大回歸演算法評測)**
5. **Evaluation (模型評估：5 大演算法 PK 排行榜 + 前向特徵選取)**
6. **Deployment (模型部署與封裝：gzip + pickle)**


## 0. 匯入必要套件


In [ ]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


## Phase 1: Business Understanding (商業理解)
- **商業目標**：分析並預測 50 家新創公司的獲利（Profit），找出各項支出（R&D, Administration, Marketing）與地理區域（State）對利潤的關鍵影響力。
- **演算法橫向評測**：比較多元線性回歸、Ridge、Lasso、Random Forest、Gradient Boosting 共 5 大回歸演算法。
- **成功標準**：高決定係數（$R^2 \ge 0.90$）以及低均方根誤差（RMSE）與平均絕對誤差（MAE）。


## Phase 2: Data Understanding (資料理解)
載入資料並檢視資料維度、統計摘要、空值分佈。


In [ ]:
data_file = "50_Startups.csv" if os.path.exists("50_Startups.csv") else "50_Startups (2).csv"
df = pd.read_csv(data_file)
print(f"資料形狀: {df.shape}")
display(df.head())
print("\n--- 欄位資訊 ---")
print(df.info())
print("\n--- 缺失值統計 ---")
print(df.isnull().sum())
print("\n--- 敘述統計 ---")
display(df.describe())


## Phase 3: Data Preparation (資料準備)
1. **分離特徵與目標**：$X$ 為前 4 個變數，$y$ 為 Profit。
2. **獨熱編碼 (One-Hot Encoding)**：針對類別欄位 `State`。
   - 設定 `drop_first=True` 避免 **Dummy Variable Trap (虛擬變數陷阱)**。
3. **資料切分 (Train/Test Split)**：80% 訓練集，20% 測試集，`random_state=0`。


In [ ]:
X = df.drop("Profit", axis=1)
y = df["Profit"]

X_encoded = pd.get_dummies(X, columns=["State"], drop_first=True, dtype=int)
print("編碼後特徵欄位:", list(X_encoded.columns))
display(X_encoded.head())

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=0)
print(f"訓練集筆數: {len(X_train)}, 測試集筆數: {len(X_test)}")


## Phase 4: Modeling & 5 大演算法橫向評測 (Benchmark)
同時訓練 5 種經典回歸模型：
1. **隨機森林 (Random Forest Regressor)**
2. **梯度提升 (Gradient Boosting Regressor)**
3. **套索回歸 (Lasso Regression, L1)**
4. **嶺回歸 (Ridge Regression, L2)**
5. **多元線性回歸 (Linear Regression, OLS)**


In [ ]:
models = {
    "隨機森林 (Random Forest)": RandomForestRegressor(n_estimators=100, random_state=0),
    "梯度提升 (Gradient Boosting)": GradientBoostingRegressor(n_estimators=100, random_state=0),
    "套索回歸 (Lasso, L1)": Lasso(alpha=100.0, random_state=0),
    "嶺回歸 (Ridge, L2)": Ridge(alpha=1.0),
    "多元線性回歸 (Linear Regression)": LinearRegression()
}

kf = KFold(n_splits=5, shuffle=True, random_state=0)
bench_records = []
fitted_models = {}

for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    fitted_models[name] = mdl
    preds = mdl.predict(X_test)
    t_r2 = r2_score(y_test, preds)
    t_rmse = np.sqrt(mean_squared_error(y_test, preds))
    t_mae = mean_absolute_error(y_test, preds)
    cv_r2 = cross_val_score(mdl, X_encoded, y, cv=kf, scoring="r2").mean()
    bench_records.append({
        "演算法 (Algorithm)": name,
        "測試集 R2": round(t_r2, 6),
        "測試集 RMSE": round(t_rmse, 2),
        "測試集 MAE": round(t_mae, 2),
        "5-Fold CV R2": round(cv_r2, 6)
    })

bench_df = pd.DataFrame(bench_records).sort_values(by="測試集 R2", ascending=False)
display(bench_df)


## Phase 5: Evaluation (模型評估與前向特徵選取)
### 1. 散布圖對比 (實際值 vs 預測值)


In [ ]:
lr_preds = fitted_models["多元線性回歸 (Linear Regression)"].predict(X_test)
rf_preds = fitted_models["隨機森林 (Random Forest)"].predict(X_test)

plt.figure(figsize=(9, 6))
plt.scatter(y_test, lr_preds, color="#3B82F6", s=80, alpha=0.8, edgecolors="k", label="Linear Regression (R2=0.9347)")
plt.scatter(y_test, rf_preds, color="#10B981", s=90, marker="^", alpha=0.9, edgecolors="k", label="Random Forest (R2=0.9699)")
min_v, max_v = min(y_test.min(), min(lr_preds.min(), rf_preds.min())), max(y_test.max(), max(lr_preds.max(), rf_preds.max()))
plt.plot([min_v, max_v], [min_v, max_v], "r--", lw=2, label="Perfect Fit (y = x)")
plt.xlabel("Actual Profit (實際利潤)", fontsize=12)
plt.ylabel("Predicted Profit (預測利潤)", fontsize=12)
plt.title("Actual vs Predicted Profit (Linear Regression vs Random Forest)", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


### 2. Forward Feature Selection (前向逐步特徵選取實驗)
由 1 個特徵遞增至 5 個特徵，驗證特徵組合的表現：


In [ ]:
def evaluate_performance(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred)), r2_score(y_true, y_pred)

def forward_selection(X_tr, X_te, y_tr, y_te, max_features):
    selected = []
    results = []
    all_cols = list(X_tr.columns)
    for i in range(1, max_features + 1):
        best_rmse, best_r2, best_feat = np.inf, -np.inf, None
        for feat in [c for c in all_cols if c not in selected]:
            curr_cols = selected + [feat]
            lr = LinearRegression().fit(X_tr[curr_cols], y_tr)
            c_rmse, c_r2 = evaluate_performance(y_te, lr.predict(X_te[curr_cols]))
            if c_rmse < best_rmse:
                best_rmse, best_r2, best_feat = c_rmse, c_r2, feat
        selected.append(best_feat)
        results.append({
            "Number of Features": i,
            "Selected Features": selected.copy(),
            "RMSE": round(best_rmse, 6),
            "R-squared": round(best_r2, 6)
        })
    return pd.DataFrame(results)

results_df = forward_selection(X_train, X_test, y_train, y_test, max_features=5)
display(results_df)


## Phase 6: Deployment (模型部署與封裝)
使用 `gzip` 配合 `pickle` 壓縮持久化存檔，並模擬部署後載入進行新創預測。


In [ ]:
model_file = "startup_lr_model.pkl.gz"
with gzip.open(model_file, "wb") as f:
    pickle.dump({"models": fitted_models, "features": list(X_encoded.columns)}, f)
print(f"模型已成功封裝並導出至: {model_file}")

sample = pd.DataFrame([{
    "R&D Spend": 160000.0,
    "Administration": 130000.0,
    "Marketing Spend": 300000.0,
    "State_Florida": 1,
    "State_New York": 0
}], columns=X_encoded.columns)

print("\n--- 範例新創公司推論預測利潤 ---")
for m_name, mdl in fitted_models.items():
    pred_val = mdl.predict(sample)[0]
    print(f"{m_name:35s}: ${pred_val:,.2f}")
